# Q1 — Marketing CPI Validation

Validate Performance Marketing's claim: **Meta CPI ~43 EGP vs Google ~94 EGP (Q3 2026)**.

Sections:
1. Config & constants
2. `marketing_spend` — validation
3. `marketing_spend` — CPI (spend-weighted)
4. `users` — signup reconciliation *(next)*

In [1]:
# CELL 1 — IMPORTS & CONFIGURATION
import glob
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# ── Paths ──
ROOT = next(folder for folder in (Path.cwd(), Path.cwd().parent) if (folder / "data").is_dir())
DATA_DIR = ROOT / "data" / "cleaned"
RAW_DIR = ROOT / "data" / "raw"
SPEND_FILE_PATTERN = "*marketing_spend.csv"
USERS_FILE_PATTERN = "*users.csv"

# ── Analysis window (Q3 2026, Jul 1 – Sep 30 inclusive) ──
Q3_START = pd.Timestamp("2026-07-01")
Q3_END   = pd.Timestamp("2026-10-01")   # exclusive upper bound
Q3_DAYS  = (Q3_END - Q3_START).days     # = 92
EXTRACT_DATE = pd.Timestamp("2026-11-05")

# ── Expected schema & invariants ──
EXPECTED_CHANNELS = ["Meta", "Google Ads", "TikTok"]
EXPECTED_ROWS_PER_DAY = len(EXPECTED_CHANNELS)   # = 3
EXPECTED_TOTAL_ROWS = Q3_DAYS * EXPECTED_ROWS_PER_DAY   # = 276
 
# ── QA / test accounts — explicit list (validated against users table repeated user_ids output) ──
QA_NAMES_EXPLICIT = [
    "QA Lucky", "Test User", "test qa", "test test",
    "تجربه تطبيق",   # Arabic: "app trial"
]

# ── Channel bucketing ──────────────────────────────────────────────
# Source is the standard attribution field. Lowercase-normalised.
SOURCE_TO_CHANNEL = {
    # Meta (7 variants)
    "meta":         "Meta",
    "facebook ads": "Meta",
    "facebook":     "Meta",
    "fb":           "Meta",
    "instagram":    "Meta",
    "ig":           "Meta",
    # Google Ads (4 variants)
    "google":       "Google Ads",
    "google ads":   "Google Ads",
    "adwords":      "Google Ads",
    "gads":         "Google Ads",
    # TikTok (3 variants)
    "tiktok":       "TikTok",
    "tt":           "TikTok",
    # Referral (4 variants)
    "referral":     "Referral",
    "ref":          "Referral",
    "invite":       "Referral",
    # Non-paid
    "organic":      "Organic",
    "(direct)":     "Direct",
}

# Campaign → channel — used as corroboration, NOT primary bucketing
CAMPAIGN_TO_CHANNEL = {
    "meta_install_q3": "Meta",
    "meta_lal_aug":    "Meta",
    "meta_retarget":   "Meta",
    "uac_q3":          "Google Ads",
    "search_brand":    "Google Ads",
    "tt_spark_q3":     "TikTok",
    "invite_friend":   "Referral",
}

PAID_CHANNELS = ["Meta", "Google Ads", "TikTok"]


# ── Display ──
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
# CELL 2 — LOAD + VALIDATE marketing_spend

spend = pd.read_csv(DATA_DIR / "marketing_spend.csv", parse_dates=["date"])

# ── Shape & schema ──
print(f"Rows: {len(spend)}  (expected {EXPECTED_TOTAL_ROWS})")
print(f"Columns: {list(spend.columns)}")

# ── Platform counts ──
print("\n── Rows per platform ──")
print(spend.channel.value_counts().to_string())

# ── Date range ──
print(f"\nDate range: {spend.date.min().date()} → {spend.date.max().date()}")
print(f"Expected  : {Q3_START.date()} → {(Q3_END - pd.Timedelta(days=1)).date()}")

# ── Date completeness (each channel × each day) ──
days_per_channel = spend.groupby("channel").date.nunique()
print("\n── Days present per channel (expected %d) ──" % Q3_DAYS)
print(days_per_channel.to_string())

# ── Duplicate (date, channel) check ──
dups = spend.duplicated(["date", "channel"]).sum()
print(f"\nDuplicate (date, channel) rows: {dups}  (expected 0)")

# ── Each calendar date appears exactly N times (once per channel) ──
rows_per_day = spend.groupby("date").size()
bad_days = rows_per_day[rows_per_day != EXPECTED_ROWS_PER_DAY]
print(f"Days with ≠ {EXPECTED_ROWS_PER_DAY} rows: {len(bad_days)}  (expected 0)")
if len(bad_days):
    print(bad_days.to_string())

Rows: 276  (expected 276)
Columns: ['date', 'channel', 'spend_egp', 'impressions', 'clicks', 'platform_reported_installs']

── Rows per platform ──
channel
Meta          92
Google Ads    92
TikTok        92

Date range: 2026-07-01 → 2026-09-30
Expected  : 2026-07-01 → 2026-09-30

── Days present per channel (expected 92) ──
channel
Google Ads    92
Meta          92
TikTok        92

Duplicate (date, channel) rows: 0  (expected 0)
Days with ≠ 3 rows: 0  (expected 0)


In [3]:
# CELL 3 — CPI (SPEND-WEIGHTED)

# ── Overall (all channels combined) ──
tot_spend    = spend.spend_egp.sum()
tot_installs = spend.platform_reported_installs.sum()
overall_cpi  = tot_spend / tot_installs

print(f"All channels — spend: {tot_spend:,.2f} EGP | installs: {tot_installs:,} | CPI: {overall_cpi:,.2f} EGP")

# ── Per channel ──
cpi = (spend
       .groupby("channel", as_index=False)
       .agg(spend_egp=("spend_egp", "sum"),
            reported_installs=("platform_reported_installs", "sum"))
       .assign(cpi_reported=lambda d: d.spend_egp / d.reported_installs)
       .sort_values("cpi_reported"))

print("\n── CPI per channel")
print(cpi.round(2).to_string(index=False))

All channels — spend: 356,538.71 EGP | installs: 6,343 | CPI: 56.21 EGP

── CPI per channel
   channel  spend_egp  reported_installs  cpi_reported
    TikTok  42,652.36               1185         35.99
      Meta 145,369.33               3355         43.33
Google Ads 168,517.02               1803         93.46


In [4]:
# =====================================================================
# SECTION 3 (continued) — integrity assertions
# =====================================================================
assert len(spend) == EXPECTED_TOTAL_ROWS,         f"Row count drift: {len(spend)}"
assert spend.duplicated(["date", "channel"]).sum() == 0, "Duplicate (date, channel)"
assert (spend.groupby("date").size() == EXPECTED_ROWS_PER_DAY).all(), "Days with ≠3 rows"
assert set(spend.channel) == set(EXPECTED_CHANNELS), f"Unexpected channels: {set(spend.channel)}"

# cross-check: per-channel sums == overall sums
assert np.isclose(cpi.spend_egp.sum(), tot_spend),        "Spend mismatch"
assert cpi.reported_installs.sum() == tot_installs,        "Install count mismatch"

print("✓ All marketing_spend integrity checks passed")

✓ All marketing_spend integrity checks passed


In [5]:
# =====================================================================
# SECTION 4 — users validation (QA removed upfront)
# =====================================================================
USERS_PATH = DATA_DIR / "users.csv"
users = pd.read_csv(USERS_PATH, parse_dates=["signup_ts", "kyc_verified_ts"])

# ── 1. File & schema ────────────────────────────────────────────────
print("── 1. File & schema ──")
print(f"File    : {Path(USERS_PATH).name}")
print(f"Rows    : {len(users)}")
print(f"Columns : {list(users.columns)}")

# ── 3. user_id ──────────────────────────────────────────────────────
print("\n── 3. user_id ──")
print(f"Distinct user_id          : {users.user_id.nunique()}")
print(f"Null user_id              : {int(users.user_id.isna().sum())}")
non_unique_ids = users.user_id[users.user_id.duplicated(keep=False)].unique().tolist()
print(f"Non-unique user_id count  : {len(non_unique_ids)}")
print(f"Non-unique user_id list   : {non_unique_ids}")

# ── 4. full_name — repeated names ───────────────────────────────────
# print("\n── 4. full_name (repeated only) ──")
# name_counts = (users.full_name.fillna("").value_counts()
#                .rename("users").reset_index())
# name_counts.columns = ["full_name", "users"]
# repeated = name_counts[(name_counts.users > 1) & (name_counts.full_name != "")]
# print(f"Repeated names count : {len(repeated)}")
# for _, row in repeated.iterrows():
#     print(f"  {row.full_name!r} : {int(row.users)}")

# ── 5. signup_ts ────────────────────────────────────────────────────
print("\n── 5. signup_ts ──")
print(f"Range                        : {users.signup_ts.min()} → {users.signup_ts.max()}")
q3_days     = pd.date_range(Q3_START, Q3_END - pd.Timedelta(days=1), freq="D")
signed_days = pd.to_datetime(users.signup_ts.dt.date).dt.normalize().unique()
print(f"Days with zero signups in Q3 : {len(q3_days) - len(signed_days)} / {len(q3_days)}")

# ── 6. utm_source (raw) ─────────────────────────────────────────────
# print("\n── 6. utm_source (raw) ──")
# src_clean = users.utm_source.fillna("(blank)").replace("", "(blank)")
# print(f"Distinct non-null values : {src_clean[src_clean != '(blank)'].nunique()}")
# print(src_clean.value_counts().rename("users").to_string())

# ── 7. utm_campaign × utm_source cross-tab ──────────────────────────
# print("\n── 7. campaign × source cross-tab ──")
# camp_clean = users.utm_campaign.fillna("(blank)").replace("", "(blank)")
# xtab = (users.assign(_src=src_clean, _camp=camp_clean)
#         .pivot_table(index="_camp", columns="_src",
#                      values="user_id", aggfunc="count", fill_value=0))
# print(xtab.to_string())
# print(f"Campaign set, source blank : "
#       f"{int(((camp_clean != '(blank)') & (src_clean == '(blank)')).sum())}")

── 1. File & schema ──
File    : users.csv
Rows    : 6200
Columns : ['user_id', 'signup_ts', 'full_name', 'governorate', 'utm_source', 'utm_campaign', 'kyc_verified_ts', 'utm_source_normalized', 'utm_campaign_normalized']

── 3. user_id ──
Distinct user_id          : 6200
Null user_id              : 0
Non-unique user_id count  : 0
Non-unique user_id list   : []

── 5. signup_ts ──
Range                        : 2026-07-01 00:22:55 → 2026-09-30 23:47:10
Days with zero signups in Q3 : 0 / 92


In [6]:
# =====================================================================
# SECTION 5 — Channel bucketing
# =====================================================================
# Rule: bucket by utm_source (standard field). Blank → "Unattributed".
# Validate against utm_campaign (must agree wherever campaign is set).

def bucket_by_source(src):
    s = str(src).strip().lower()
    if s in ("", "nan", "none"):
        return "Unattributed"
    return SOURCE_TO_CHANNEL.get(s, "Unknown")

users["channel"] = users.utm_source.map(bucket_by_source)

# ── Validation: campaign vs source must agree wherever campaign is set ──
camp_derived = users.utm_campaign.str.strip().map(CAMPAIGN_TO_CHANNEL)
comparable   = camp_derived.notna() & users.channel.notna()
mismatches   = (camp_derived[comparable] != users.channel[comparable]).sum()
unknown      = (users.channel == "Unknown").sum()
print(f"Campaign vs source mismatches : {mismatches}  (expected 0)")
print(f"Unknown source values         : {unknown}     (expected 0)")

# ── Bucket counts ───────────────────────────────────────────────────
counts = users.channel.value_counts().rename("users")
pct    = (100 * counts / counts.sum()).round(1).rename("pct")
summary = pd.concat([counts, pct], axis=1).sort_values("users", ascending=False)

print("\n── Channel buckets (after QA removal) ──")
print(f"Total users : {len(users)}")
print(summary.to_string())

# integrity
assert summary.users.sum() == len(users), "Bucket sum ≠ user count"
assert unknown == 0, "Unrecognised source fell through — extend SOURCE_TO_CHANNEL"
assert mismatches == 0, "Campaign/source disagreement — investigate before proceeding"
print(f"\n✓ {len(users)} users bucketed · {len(PAID_CHANNELS)} paid channels · "
      f"{summary.loc['Unattributed','users'] if 'Unattributed' in summary.index else 0} unattributed")

Campaign vs source mismatches : 0  (expected 0)
Unknown source values         : 0     (expected 0)

── Channel buckets (after QA removal) ──
Total users : 6200
              users   pct
channel                  
Meta           2100 33.90
Google Ads     1430 23.10
Unattributed    764 12.30
Referral        741 12.00
TikTok          650 10.50
Organic         264  4.30
Direct          251  4.00

✓ 6200 users bucketed · 3 paid channels · 764 unattributed


In [7]:
paid_signups    = (users[users.channel.isin(PAID_CHANNELS)]
                   .groupby("channel").size()
                   .reindex(PAID_CHANNELS).fillna(0).astype(int))
n_unattributed  = int((users.channel == "Unattributed").sum())
spend_by_channel = cpi.set_index("channel").loc[PAID_CHANNELS, "spend_egp"]

signups_strict      = paid_signups.copy()
signups_all_google  = paid_signups.copy(); signups_all_google["Google Ads"] += n_unattributed
signups_all_meta    = paid_signups.copy(); signups_all_meta["Meta"]         += n_unattributed
signups_split       = (paid_signups + paid_signups / paid_signups.sum() * n_unattributed).round().astype(int)

result = (pd.DataFrame({
            "spend_egp"  : spend_by_channel.round(2),
            "strict"     : (spend_by_channel / signups_strict    ).round(2),
            "all_google" : (spend_by_channel / signups_all_google).round(2),
            "all_meta"   : (spend_by_channel / signups_all_meta  ).round(2),
            "split"      : (spend_by_channel / signups_split     ).round(2),
          })
          .rename_axis("channel")
          .reset_index())

print(f"Unattributed distributed: {n_unattributed} users\n")
result

Unattributed distributed: 764 users

Out[0]: 
      channel  spend_egp  strict  all_google  all_meta  split
0        Meta 145,369.33   69.22       69.22     50.76  58.52
1  Google Ads 168,517.02  117.84       76.81    117.84  99.66
2      TikTok  42,652.36   65.62       65.62     65.62  55.46


Google is 100% the highest cpi confirmed where at absolute best assumption it is still costing 7 egp per install compared to meta

In [8]:
# =====================================================================
# SECTION 7A — Saturation quartiles + spend totals & distribution
# =====================================================================


SATURATION_FLAT_PCT     = 5.0
SATURATION_MODERATE_PCT = 10.0


try:
    q3_daily
except NameError:
    _spend = pd.read_csv(DATA_DIR / "marketing_spend.csv", parse_dates=["date"])
    q3_daily = _spend.query("date >= @Q3_START and date < @Q3_END").copy()
    q3_daily["cpi_daily"] = q3_daily.spend_egp / q3_daily.platform_reported_installs

def classify_saturation(delta_pct):
    if delta_pct < SATURATION_FLAT_PCT:
        return "Not saturated — scales well"
    elif delta_pct < SATURATION_MODERATE_PCT:
        return "Partially saturated — scales ok"
    else:
        return "Saturated — will not scale"

total_q3_spend = q3_daily.spend_egp.sum()

rows = []
for ch in EXPECTED_CHANNELS:
    sub = q3_daily[q3_daily.channel == ch].copy()
    sub["spend_quartile"] = pd.qcut(
        sub.spend_egp, 4,
        labels=["Lowest-spend 25%", "Low-mid", "High-mid", "Highest-spend 25%"]
    )
    g = (sub.groupby("spend_quartile", observed=True)
            .agg(median_spend=("spend_egp", "median"),
                 median_cpi  =("cpi_daily",  "median"),
                 total_spend =("spend_egp", "sum")))
    low, high = g.iloc[0], g.iloc[-1]

    delta_egp = high.median_cpi - low.median_cpi
    delta_pct = 100 * delta_egp / low.median_cpi

    channel_total = sub.spend_egp.sum()

    rows.append({
        "channel"                       : ch,
        "q3_total_spend"                : channel_total,
        "pct_of_total_spend"            : 100 * channel_total / total_q3_spend,
        "low_spend_median_egp_per_day"  : low.median_spend,
        "low_spend_median_cpi"          : low.median_cpi,
        "low_quartile_pct_of_channel"   : 100 * low.total_spend / channel_total,
        "high_spend_median_egp_per_day" : high.median_spend,
        "high_spend_median_cpi"         : high.median_cpi,
        "high_quartile_pct_of_channel"  : 100 * high.total_spend / channel_total,
        "cpi_change_egp"                : delta_egp,
        "cpi_change_pct"                : delta_pct,
        "bucket"                        : classify_saturation(delta_pct),
    })

saturation = pd.DataFrame(rows).round(2)
saturation

,channel,q3_total_spend,pct_of_total_spend,low_spend_median_egp_per_day,low_spend_median_cpi,low_quartile_pct_of_channel,high_spend_median_egp_per_day,high_spend_median_cpi,high_quartile_pct_of_channel,cpi_change_egp,cpi_change_pct,bucket
0,Meta,"145,369.33",40.77,786.00,42.85,12.24,"2,696.15",43.63,42.02,0.78,1.82,Not saturated — scales well
1,Google Ads,"168,517.02",47.26,"1,213.62",84.11,16.61,"2,418.07",98.57,34.76,14.46,17.20,Saturated — will not scale
2,TikTok,"42,652.36",11.96,213.65,34.93,10.17,739.46,37.06,42.54,2.13,6.09,Partially saturated — scales ok


In [9]:
# =====================================================================
# SECTION 7B — December split: buckets → recommended allocation
# =====================================================================
DEC_BUDGET = 150_000
DEC_SHARES = {"Google Ads": 0.35, "Meta": 0.45, "TikTok": 0.20}

# Q3 monthly-average spend per channel (baseline for comparison)
q3_monthly = (q3_daily.groupby("channel").spend_egp.sum() / 3).round(0)

# Justification strings — one per channel, tied to its bucket
JUSTIFICATION = {
    "Google Ads": "Saturated — cut. Keep 35% floor to preserve Search-brand (high intent).",
    "Meta":       "Not saturated — scale. 45% cap avoids >50% single-platform concentration.",
    "TikTok":     "Partially saturated — scale modestly. Small base means even 20% is a test.",
}

split = pd.DataFrame({
    "channel"      : PAID_CHANNELS,
    "bucket"       : saturation.set_index("channel").loc[PAID_CHANNELS, "bucket"],
    "q3_monthly"   : q3_monthly.loc[PAID_CHANNELS].values,
    "dec_share_pct": [100 * DEC_SHARES[c]      for c in PAID_CHANNELS],
    "dec_egp"      : [DEC_BUDGET * DEC_SHARES[c] for c in PAID_CHANNELS],
    "justification": [JUSTIFICATION[c] for c in PAID_CHANNELS],
}).assign(
    dec_vs_q3_pct = lambda d: (100 * (d.dec_egp / d.q3_monthly - 1)).round(1),
)

split

,channel,bucket,q3_monthly,dec_share_pct,dec_egp,justification,dec_vs_q3_pct
channel,,,,,,,
Meta,Meta,Not saturated — scales well,"48,456.00",45.00,"67,500.00",Not saturated — scale. 45% cap avoids >50% sin...,39.30
Google Ads,Google Ads,Saturated — will not scale,"56,172.00",35.00,"52,500.00",Saturated — cut. Keep 35% floor to preserve Se...,-6.50
TikTok,TikTok,Partially saturated — scales ok,"14,217.00",20.00,"30,000.00",Partially saturated — scale modestly. Small ba...,111.00


In [10]:
# =====================================================================
# SECTION 7C — Scenario comparison at 150K: per-channel CPI, no blended
# =====================================================================

# Observed Q3 baselines
total_q3     = q3_daily.spend_egp.sum()
Q3_SHARES    = (q3_daily.groupby("channel").spend_egp.sum() / total_q3).to_dict()
Q3_CPI       = cpi.set_index("channel")["cpi_reported"].to_dict()
q3_installs  = int(q3_daily.platform_reported_installs.sum())

def project_by_channel(shares, budget):
    """Return per-channel dicts: {ch: (egp, cpi, installs)}."""
    out = {}
    for ch in PAID_CHANNELS:
        egp      = budget * shares[ch]
        cpi_val  = Q3_CPI[ch]                # constant CPI assumption
        installs = egp / cpi_val
        out[ch]  = {"egp": egp, "cpi": cpi_val, "installs": installs}
    return out

q3_proj  = project_by_channel(Q3_SHARES,  DEC_BUDGET)
rec_proj = project_by_channel(DEC_SHARES, DEC_BUDGET)

q3_at_150_total  = sum(v["installs"] for v in q3_proj.values())
rec_at_150_total = sum(v["installs"] for v in rec_proj.values())
delta_installs   = int(round(rec_at_150_total - q3_at_150_total))
delta_pct        = 100 * (rec_at_150_total / q3_at_150_total - 1)

scenarios = pd.DataFrame([
    {
        "scenario"   : "Q3 actual (history)",
        "budget_egp" : int(round(total_q3)),
        "meta_pct"   : round(100 * Q3_SHARES["Meta"], 1),
        "meta_cpi"   : Q3_CPI["Meta"],
        "google_pct" : round(100 * Q3_SHARES["Google Ads"], 1),
        "google_cpi" : Q3_CPI["Google Ads"],
        "tiktok_pct" : round(100 * Q3_SHARES["TikTok"], 1),
        "tiktok_cpi" : Q3_CPI["TikTok"],
        "installs"   : q3_installs,
        "delta_vs_baseline": "—",
    },
    {
        "scenario"   : "Q3 shares @ 150K (do-nothing)",
        "budget_egp" : DEC_BUDGET,
        "meta_pct"   : round(100 * Q3_SHARES["Meta"], 1),
        "meta_cpi"   : Q3_CPI["Meta"],
        "google_pct" : round(100 * Q3_SHARES["Google Ads"], 1),
        "google_cpi" : Q3_CPI["Google Ads"],
        "tiktok_pct" : round(100 * Q3_SHARES["TikTok"], 1),
        "tiktok_cpi" : Q3_CPI["TikTok"],
        "installs"   : int(round(q3_at_150_total)),
        "delta_vs_baseline": "— (baseline)",
    },
    {
        "scenario"   : "Recommended 45/35/20 @ 150K",
        "budget_egp" : DEC_BUDGET,
        "meta_pct"   : round(100 * DEC_SHARES["Meta"], 1),
        "meta_cpi"   : Q3_CPI["Meta"],
        "google_pct" : round(100 * DEC_SHARES["Google Ads"], 1),
        "google_cpi" : Q3_CPI["Google Ads"],
        "tiktok_pct" : round(100 * DEC_SHARES["TikTok"], 1),
        "tiktok_cpi" : Q3_CPI["TikTok"],
        "installs"   : int(round(rec_at_150_total)),
        "delta_vs_baseline": f"+{delta_installs} ({delta_pct:+.1f}%)",
    },
])

scenarios

,scenario,budget_egp,meta_pct,meta_cpi,google_pct,google_cpi,tiktok_pct,tiktok_cpi,installs,delta_vs_baseline
0,Q3 actual (history),356539,40.80,43.33,47.30,93.46,12.00,35.99,6343,—
1,Q3 shares @ 150K (do-nothing),150000,40.80,43.33,47.30,93.46,12.00,35.99,2669,— (baseline)
2,Recommended 45/35/20 @ 150K,150000,45.00,43.33,35.00,93.46,20.00,35.99,2953,+284 (+10.7%)


In [11]:
# =====================================================================
# SECTION 8A — DQ on transactions
# =====================================================================
txn = pd.read_csv(next(RAW_DIR.glob("* - transactions.csv")), parse_dates=["txn_ts"])

print(f"Rows         : {len(txn):,}")
print(f"Unique txn_id: {txn.txn_id.nunique():,}")
print(f"Duplicate txn_id rows: {int(txn.txn_id.duplicated().sum()):,}")

# Compare against raw IDs to distinguish orphans from known QA accounts.
raw_user_ids = pd.read_csv(next(RAW_DIR.glob("* - users.csv")), usecols=["user_id"])["user_id"]
orphans = txn[~txn.user_id.isin(raw_user_ids)]
print(f"\nOrphan txn rows (user not in raw users): {len(orphans):,}")
print(f"Orphan distinct users              : {orphans.user_id.nunique():,}")

# Dupes: are they exact duplicates?
dup_rows = txn[txn.txn_id.duplicated(keep=False)].sort_values("txn_id")
exact_dups = dup_rows.drop_duplicates().shape[0]
print(f"\nDuplicate-txn_id rows: {len(dup_rows):,}  |  after full dedupe: {exact_dups:,}")
print(f"Exact duplicates (all cols match): {len(dup_rows) - exact_dups:,}")

# Status + type distribution on successful only
print(f"\nSuccessful rows  : {int((txn.status == 'success').sum()):,}")
print(f"Failed rows      : {int((txn.status == 'failed').sum()):,}")

Rows         : 42,378
Unique txn_id: 41,801
Duplicate txn_id rows: 577

Orphan txn rows (user not in raw users): 0
Orphan distinct users              : 0

Duplicate-txn_id rows: 1,154  |  after full dedupe: 577
Exact duplicates (all cols match): 577

Successful rows  : 39,059
Failed rows      : 3,319


In [12]:
# =====================================================================
# SECTION 8B — Activated users (README definition)
# KYC verified AND ≥1 successful deposit ≤7d of signup.
# =====================================================================
clean_transactions = pd.read_csv(DATA_DIR / "transactions.csv", parse_dates=["txn_ts"])

txn_clean = (clean_transactions
    .query("status == 'success' and txn_type == 'deposit'")   # deposits only
    .merge(users[["user_id", "signup_ts", "kyc_verified_ts", "channel"]],
           on="user_id", how="inner")
    .copy()
)

txn_clean["days_since_signup"] = (txn_clean.txn_ts - txn_clean.signup_ts).dt.total_seconds() / 86400
deposit_within_7d = txn_clean[txn_clean.days_since_signup.between(0, 7)].copy()

# Users who deposited ≤7d
deposited_users = deposit_within_7d.user_id.unique()
print(f"Users with deposit ≤7d         : {len(deposited_users):,}")

# Of those, how many were KYC verified?
kyc_ok = deposit_within_7d[deposit_within_7d.kyc_verified_ts.notna()].user_id.unique()
print(f"of which KYC verified          : {len(kyc_ok):,}  ← README 'activated'")
print(f"KYC coverage among depositors  : {100*len(kyc_ok)/len(deposited_users):.1f}%")

# Build activation table per channel
activated = (deposit_within_7d[deposit_within_7d.kyc_verified_ts.notna()]
             .drop_duplicates("user_id")
             .groupby("channel").size()
             .reindex(PAID_CHANNELS).fillna(0).astype(int))
print(f"\nActivated users by channel:")
print(activated.to_string())

Users with deposit ≤7d         : 2,361
of which KYC verified          : 2,361  ← README 'activated'
KYC coverage among depositors  : 100.0%

Activated users by channel:
channel
Meta          522
Google Ads    730
TikTok        132


In [13]:
# =====================================================================
# SECTION 8C — Metrics ladder: CPI → cost/signup → cost/activated → cost/first-txn
# =====================================================================
# cost/activated   = spend / README-activated count (KYC + deposit ≤7d)
# cost/first_txn   = spend / any successful txn ≤7d (my earlier definition)
# =====================================================================

# First-txn count (any type) — recalculated for comparison
any_first = (clean_transactions.query("status == 'success'")
             .merge(users[["user_id", "signup_ts", "channel"]], on="user_id", how="inner")
             .assign(days_since_signup=lambda d: (d.txn_ts - d.signup_ts).dt.total_seconds()/86400)
             .query("days_since_signup >= 0 and days_since_signup <= 7")
             .sort_values(["user_id", "txn_ts"])
             .drop_duplicates("user_id", keep="first"))
any_first_count = (any_first[any_first.channel.isin(PAID_CHANNELS)]
                   .groupby("channel").size()
                   .reindex(PAID_CHANNELS).fillna(0).astype(int))

paid_signups = (users[users.channel.isin(PAID_CHANNELS)].groupby("channel").size()
                .reindex(PAID_CHANNELS).fillna(0).astype(int))
spend_by_ch  = cpi.set_index("channel").loc[PAID_CHANNELS, "spend_egp"]

ladder = pd.DataFrame({
    "channel"             : PAID_CHANNELS,
    "reported_installs"   : cpi.set_index("channel").loc[PAID_CHANNELS, "reported_installs"].values,
    "signups"             : paid_signups.values,
    "activated_README"    : activated.values,
    "first_txn_any_type"  : any_first_count.values,
    "spend_egp"           : spend_by_ch.values,
}).set_index("channel")

ladder["CPI"]                   = (ladder.spend_egp / ladder.reported_installs).round(2)
ladder["cost_per_signup"]       = (ladder.spend_egp / ladder.signups).round(2)
ladder["cost_per_activated"]    = (ladder.spend_egp / ladder.activated_README).round(2)
ladder["cost_per_first_txn"]    = (ladder.spend_egp / ladder.first_txn_any_type).round(2)
ladder["activation_rate_pct"]   = (100 * ladder.activated_README / ladder.signups).round(1)

ladder[["CPI", "cost_per_signup", "cost_per_activated", "cost_per_first_txn",
        "signups", "activated_README", "activation_rate_pct"]]

,CPI,cost_per_signup,cost_per_activated,cost_per_first_txn,signups,activated_README,activation_rate_pct
channel,,,,,,,
Meta,43.33,69.22,278.49,278.49,2100,522,24.90
Google Ads,93.46,117.84,230.85,230.85,1430,730,51.00
TikTok,35.99,65.62,323.12,323.12,650,132,20.30


In [15]:
# =====================================================================
# SECTION 9 — Does activation rate degrade at high spend per channel?
# =====================================================================
# If Google's 51% holds across spend quartiles → Google scales on activated
# If it falls at high spend → saturation bites at customer level

# Daily activated users by channel
daily_activated = (deposit_within_7d[deposit_within_7d.kyc_verified_ts.notna()]
                   .drop_duplicates(["user_id"])
                   .assign(signup_date=lambda d: d.signup_ts.dt.normalize())
                   .groupby(["signup_date", "channel"]).size()
                   .rename("activated_users").reset_index())

# Join to daily spend (from q3_daily)
daily = (q3_daily[["date", "channel", "spend_egp"]]
         .merge(daily_activated, left_on=["date", "channel"],
                right_on=["signup_date", "channel"], how="left")
         .assign(activated_users=lambda d: d.activated_users.fillna(0).astype(int)))

rows = []
for ch in PAID_CHANNELS:
    sub = daily[daily.channel == ch].copy()
    sub["spend_q"] = pd.qcut(sub.spend_egp, 4,
                             labels=["Lowest-spend 25%", "Low-mid", "High-mid", "Highest-spend 25%"])
    g = (sub.groupby("spend_q", observed=True)
            .agg(mean_spend=("spend_egp", "mean"),
                 mean_activated=("activated_users", "mean")))
    low, high = g.iloc[0], g.iloc[-1]
    rows.append({
        "channel"                    : ch,
        "low_spend_avg_egp"          : round(low.mean_spend, 0),
        "low_spend_avg_activated"    : round(low.mean_activated, 2),
        "high_spend_avg_egp"         : round(high.mean_spend, 0),
        "high_spend_avg_activated"   : round(high.mean_activated, 2),
        "activated_change_pct"       : round(100 * (high.mean_activated / low.mean_activated - 1), 1),
    })

activation_sat = pd.DataFrame(rows)
activation_sat

,channel,low_spend_avg_egp,low_spend_avg_activated,high_spend_avg_egp,high_spend_avg_activated,activated_change_pct
0,Meta,774.00,3.04,"2,656.00",9.48,211.40
1,Google Ads,"1,217.00",5.39,"2,547.00",10.48,94.40
2,TikTok,189.00,0.48,789.00,2.61,445.50
